In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.predict import Predictor, ThresholdItemPredictor
from utils.split import FrequencyBandSplitter, FrequencyBandEvaluator
from model.PMF import MatrixFactorization
from model.BPMF import BayesianPMF, BayesianPMFWithBiases
from model.NNBPMF import BayesianNonNegativeMF
from ensemble.ensemble import RatingEnsemble
from ensemble.adaptative import AdaptiveColdStartEnsemble, AdaptivePosteriorColdStartEnsemble
from sklearn.model_selection import train_test_split
import random

from model.PMF import SurpriseNMFModel
from model.baseline import SurpriseBaselineOnlyModel, SurpriseKNNBaselineWrapper

In [2]:
TRAIN_PATH = '../data/train.csv'

full_data = pd.read_csv(TRAIN_PATH)

train_data, test_data = train_test_split(full_data, test_size=0.1, random_state=42)

train_data

,user,item,rating
65957,8352,155705,6.0
36595,4201,103532,9.0
74297,9619,168399,7.0
307818,59525,29423,8.0
90711,12518,29534,8.0
...,...,...,...
259178,48916,133274,7.0
365838,72117,177149,8.0
131932,21683,38124,8.0
146867,25312,26784,10.0


In [ ]:
nn_bpmf = BayesianNonNegativeMF(n_factors=8, n_iters=150, alpha=0.3, beta=5.0)
nn_bpmf.fit(train_data)

predictions_nn_bpmf = nn_bpmf.predict_df(test_data)
mae_nn_bpmf = np.mean(np.abs(predictions_nn_bpmf - test_data['rating']))
print(f'MAE for Bayesian Non-Negative Matrix Factorization: {mae_nn_bpmf}')

splitter = FrequencyBandSplitter(entity="item", low_threshold=2, high_threshold=12)
splitter.fit(train_data)
evaluator = FrequencyBandEvaluator(splitter)
mae_nn_bpmf_bands = evaluator.evaluate(nn_bpmf, test_data)
print("MAE for Bayesian Non-Negative Matrix Factorization by frequency band:")
print(mae_nn_bpmf_bands)

MAE for Bayesian Non-Negative Matrix Factorization by frequency band:
   band  n_rows      rmse       mae
0  high   21563  2.527454  2.065900
1   low   17473  3.098047  2.641561
2   mid       0       NaN       NaN


In [19]:
nn_bpmf.a_.mean(), nn_bpmf.a_.std(), nn_bpmf.b_.mean(), nn_bpmf.b_.std()

(0.125, 0.14659666777497707, 0.5254089623059853, 0.06863430843545647)

In [ ]:
nn_bpmf_2 = BayesianNonNegativeMF(n_factors=10, n_iters=150, alpha=0.1, beta=1.0)
nn_bpmf_2.fit(train_data)

predictions_nn_bpmf_2 = nn_bpmf_2.predict_df(test_data, round_predictions=True)
mae_nn_bpmf_2 = np.mean(np.abs(predictions_nn_bpmf_2['prediction'] - test_data['rating']))
print(f'MAE for Bayesian Non-Negative Matrix Factorization: {mae_nn_bpmf_2}')

splitter = FrequencyBandSplitter(entity="item", low_threshold=2, high_threshold=12)
splitter.fit(train_data)
evaluator = FrequencyBandEvaluator(splitter)
mae_nn_bpmf_2_bands = evaluator.evaluate(nn_bpmf_2, test_data)
print("MAE for Bayesian Non-Negative Matrix Factorization by frequency band:")
print(mae_nn_bpmf_2_bands)

MAE for Bayesian Non-Negative Matrix Factorization: 2.2951122041192744
MAE for Bayesian Non-Negative Matrix Factorization by frequency band:
   band  n_rows      rmse       mae
0  high   21563  2.513019  2.011037
1   low   17473  3.102477  2.645682
2   mid       0       NaN       NaN


In [29]:
nn_bpmf_2.a_.mean(), nn_bpmf_2.a_.std(), nn_bpmf_2.b_.mean(), nn_bpmf_2.b_.std()

(0.10000000000000005,
 0.19141589056173905,
 0.5346732062194728,
 0.10040979167082707)

In [33]:
# get the items with lower frequency in the test set
item_counts = test_data['item'].value_counts()
low_freq_items = item_counts[item_counts == 1].index

for item in low_freq_items[:10]:
    item_data = test_data[test_data['item'] == item]
    # select a random row from item_data
    random_rows= item_data.sample(n=1, random_state=42)
    for index, row in random_rows.iterrows():
        user_id = row['user']
        true_rating = row['rating']
        pred_nnbpmf = nn_bpmf.predict_expected_rating(user_id, item)
        print(f"User: {user_id}, Item: {item}, True Rating: {true_rating}, Predicted Rating (NNBPMF): {pred_nnbpmf:.2f}")
        reason = nn_bpmf.explain_prediction(user_id, item)
        print(f"Explanation for User: {user_id}, Item: {item}: {reason}\n")

User: 73033.0, Item: 4647, True Rating: 7.0, Predicted Rating (NNBPMF): 5.92
Explanation for User: 73033.0, Item: 4647: [{'factor': 6, 'user_prob_group': 0.125, 'item_like_prob_given_group': 0.7666497834651355, 'contribution_to_p_ui': 0.09583122293314193}, {'factor': 3, 'user_prob_group': 0.125, 'item_like_prob_given_group': 0.7288823214973473, 'contribution_to_p_ui': 0.09111029018716842}, {'factor': 2, 'user_prob_group': 0.125, 'item_like_prob_given_group': 0.649681228349875, 'contribution_to_p_ui': 0.08121015354373437}, {'factor': 5, 'user_prob_group': 0.125, 'item_like_prob_given_group': 0.5796036219442644, 'contribution_to_p_ui': 0.07245045274303305}, {'factor': 7, 'user_prob_group': 0.125, 'item_like_prob_given_group': 0.5066164768439827, 'contribution_to_p_ui': 0.06332705960549784}]

User: 39636.0, Item: 98293, True Rating: 6.0, Predicted Rating (NNBPMF): 5.04
Explanation for User: 39636.0, Item: 98293: [{'factor': 2, 'user_prob_group': 0.9615160628170748, 'item_like_prob_given_g

In [36]:
# 1) cuántos usuarios están casi uniformes
uniform = np.full(nn_bpmf.n_factors, 1.0 / nn_bpmf.n_factors)
user_l1_to_uniform = np.abs(nn_bpmf.a_ - uniform).sum(axis=1)

print("Usuarios casi uniformes (<0.05):", np.mean(user_l1_to_uniform < 0.05))
print("Usuarios casi uniformes (<0.10):", np.mean(user_l1_to_uniform < 0.10))

# 2) cuántos items están casi en 0.5
item_mean_abs_from_half = np.abs(nn_bpmf.b_ - 0.5).mean(axis=1)
print("Items casi neutros (<0.01):", np.mean(item_mean_abs_from_half < 0.01))
print("Items casi neutros (<0.03):", np.mean(item_mean_abs_from_half < 0.03))

# 3) distribución de predicciones normalizadas
pred_norm_all = []
for u, i in zip(test_data["user"], test_data["item"]):
    pred_norm_all.append(nn_bpmf.predict_normalized(u, i))
pred_norm_all = np.array(pred_norm_all)

print("Pred norm mean:", pred_norm_all.mean())
print("Pred norm std:", pred_norm_all.std())
print("Frac en [0.45, 0.55]:", np.mean((pred_norm_all >= 0.45) & (pred_norm_all <= 0.55)))

Usuarios casi uniformes (<0.05): 0.06557210611049066
Usuarios casi uniformes (<0.10): 0.07505765297982508
Items casi neutros (<0.01): 0.18219637608645028
Items casi neutros (<0.03): 0.6649862482613649
Pred norm mean: 0.5806259662877057
Pred norm std: 0.10311971622832193
Frac en [0.45, 0.55]: 0.6416897223076135


In [38]:
eval_df = test_data.copy()
eval_df["pred"] = nn_bpmf_2.predict_df(test_data)['prediction']

item_conf = []
for item in eval_df["item"]:
    if item in nn_bpmf_2.item_to_idx_:
        b_i = nn_bpmf_2.b_[nn_bpmf_2.item_to_idx_[item]]
        conf = 2.0 * np.mean(np.abs(b_i - 0.5))
    else:
        conf = 0.0
    item_conf.append(conf)

eval_df["item_conf"] = item_conf
eval_df["item_conf_bin"] = pd.cut(
    eval_df["item_conf"],
    bins=[-1e-9, 0.02, 0.05, 0.10, 1.0],
    labels=["muy_baja", "baja", "media", "alta"]
)

print(eval_df.groupby("item_conf_bin").apply(lambda g: np.mean(np.abs(g["pred"] - g["rating"]))))
print(eval_df["item_conf_bin"].value_counts())

item_conf_bin
muy_baja    2.639116
baja        2.439273
media       2.660277
alta        1.934641
dtype: float64
item_conf_bin
alta        18819
muy_baja    13938
media        4904
baja         1375
Name: count, dtype: int64


C:\Users\mario\AppData\Local\Temp\ipykernel_31092\1158731502.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(eval_df.groupby("item_conf_bin").apply(lambda g: np.mean(np.abs(g["pred"] - g["rating"]))))
C:\Users\mario\AppData\Local\Temp\ipykernel_31092\1158731502.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(eval_df.groupby("item_conf_bin").apply(lambda g: np.mean(np.abs(g["pred"] - g["rating"]))))


In [35]:
mf_10 = MatrixFactorization(n_factors=10, n_epochs=100, lr=0.01, reg=0.3)

mf_10.fit(train_data)
predictions_mf_10 = mf_10.predict_df(test_data, round_predictions=True)
mae_mf_10 = np.mean(np.abs(predictions_mf_10['prediction'] - test_data['rating']))
print(f'MAE for Matrix Factorization with 10 factors: {mae_mf_10}')

splitter = FrequencyBandSplitter(entity="item", low_threshold=2, high_threshold=12)
splitter.fit(train_data)
evaluator = FrequencyBandEvaluator(splitter)
mae_mf_10_bands = evaluator.evaluate(mf_10, test_data)
print("MAE for Matrix Factorization with 10 factors by frequency band:")
print(mae_mf_10_bands)

[MatrixFactorization] Epoch 1/100 - RMSE: 1.7518
[MatrixFactorization] Epoch 2/100 - RMSE: 1.6693
[MatrixFactorization] Epoch 3/100 - RMSE: 1.6237
[MatrixFactorization] Epoch 4/100 - RMSE: 1.5895
[MatrixFactorization] Epoch 5/100 - RMSE: 1.5610
[MatrixFactorization] Epoch 6/100 - RMSE: 1.5363
[MatrixFactorization] Epoch 7/100 - RMSE: 1.5143
[MatrixFactorization] Epoch 8/100 - RMSE: 1.4946
[MatrixFactorization] Epoch 9/100 - RMSE: 1.4762
[MatrixFactorization] Epoch 10/100 - RMSE: 1.4590
[MatrixFactorization] Epoch 11/100 - RMSE: 1.4429
[MatrixFactorization] Epoch 12/100 - RMSE: 1.4277
[MatrixFactorization] Epoch 13/100 - RMSE: 1.4130
[MatrixFactorization] Epoch 14/100 - RMSE: 1.3991
[MatrixFactorization] Epoch 15/100 - RMSE: 1.3855
[MatrixFactorization] Epoch 16/100 - RMSE: 1.3725
[MatrixFactorization] Epoch 17/100 - RMSE: 1.3596
[MatrixFactorization] Epoch 18/100 - RMSE: 1.3471
[MatrixFactorization] Epoch 19/100 - RMSE: 1.3347
[MatrixFactorization] Epoch 20/100 - RMSE: 1.3226
[MatrixFa

In [41]:
adaptative_enseble = AdaptiveColdStartEnsemble(main_model=mf_10)
adaptative_enseble.fit(train_data)

adaptative_enseble_predictions = adaptative_enseble.predict_df(test_data, round_predictions=True)
mae_adaptative_ensemble = np.mean(np.abs(adaptative_enseble_predictions['prediction'] - test_data['rating']))
print(f'MAE for Adaptive Cold Start Ensemble: {mae_adaptative_ensemble}')

[MatrixFactorization] Epoch 1/100 - RMSE: 1.7518
[MatrixFactorization] Epoch 2/100 - RMSE: 1.6693
[MatrixFactorization] Epoch 3/100 - RMSE: 1.6237
[MatrixFactorization] Epoch 4/100 - RMSE: 1.5895
[MatrixFactorization] Epoch 5/100 - RMSE: 1.5610
[MatrixFactorization] Epoch 6/100 - RMSE: 1.5363
[MatrixFactorization] Epoch 7/100 - RMSE: 1.5143
[MatrixFactorization] Epoch 8/100 - RMSE: 1.4946
[MatrixFactorization] Epoch 9/100 - RMSE: 1.4762
[MatrixFactorization] Epoch 10/100 - RMSE: 1.4590
[MatrixFactorization] Epoch 11/100 - RMSE: 1.4429
[MatrixFactorization] Epoch 12/100 - RMSE: 1.4277
[MatrixFactorization] Epoch 13/100 - RMSE: 1.4130
[MatrixFactorization] Epoch 14/100 - RMSE: 1.3991
[MatrixFactorization] Epoch 15/100 - RMSE: 1.3855
[MatrixFactorization] Epoch 16/100 - RMSE: 1.3725
[MatrixFactorization] Epoch 17/100 - RMSE: 1.3596
[MatrixFactorization] Epoch 18/100 - RMSE: 1.3471
[MatrixFactorization] Epoch 19/100 - RMSE: 1.3347
[MatrixFactorization] Epoch 20/100 - RMSE: 1.3226
[MatrixFa

In [42]:
evaluator.evaluate(adaptative_enseble, test_data)

,band,n_rows,rmse,mae
0,high,21563,1.620212,1.271180
1,low,17473,1.703354,1.319236
2,mid,0,NaN,NaN


In [17]:
bpmf_with_biases = BayesianPMFWithBiases(n_factors=10, n_iters=50, burn_in=20, rating_std=1, user_bias_std=0.5, item_bias_std=0.5, clip_range=(0, 10))
bpmf_with_biases.fit(train_data)

bpmf_with_biases_predictions = bpmf_with_biases.predict_df(test_data, round_predictions=True)
mae_bpmf_with_biases = np.mean(np.abs(bpmf_with_biases_predictions['prediction'] - test_data['rating']))
print(f'MAE for BPMF with Biases Model: {mae_bpmf_with_biases}')

evaluator.evaluate(bpmf_with_biases, test_data)

[BayesianPMFWithBiases] iter 1/50 - train_rmse=1.38284
[BayesianPMFWithBiases] iter 5/50 - train_rmse=1.38671
[BayesianPMFWithBiases] iter 10/50 - train_rmse=1.38780
[BayesianPMFWithBiases] iter 15/50 - train_rmse=1.38598
[BayesianPMFWithBiases] iter 20/50 - train_rmse=1.38400
[BayesianPMFWithBiases] iter 25/50 - train_rmse=1.38305
[BayesianPMFWithBiases] iter 30/50 - train_rmse=1.38090
[BayesianPMFWithBiases] iter 35/50 - train_rmse=1.37973
[BayesianPMFWithBiases] iter 40/50 - train_rmse=1.37928
[BayesianPMFWithBiases] iter 45/50 - train_rmse=1.37803
[BayesianPMFWithBiases] iter 50/50 - train_rmse=1.37661
MAE for BPMF with Biases Model: 1.2366277282508453


,band,n_rows,rmse,mae
0,high,11425,1.567889,1.213964
1,low,17473,1.680130,1.290903
2,mid,10138,1.627399,1.257858


In [24]:
bpmf_with_biases = BayesianPMFWithBiases(n_factors=8, n_iters=50, burn_in=20, rating_std=1, user_bias_std=0.8, item_bias_std=0.8, clip_range=(0, 10))
bpmf_with_biases.fit(train_data)

bpmf_with_biases_predictions = bpmf_with_biases.predict_df(test_data, round_predictions=True)
mae_bpmf_with_biases = np.mean(np.abs(bpmf_with_biases_predictions['prediction'] - test_data['rating']))
print(f'MAE for BPMF with Biases Model: {mae_bpmf_with_biases}')

evaluator.evaluate(bpmf_with_biases, test_data)

[BayesianPMFWithBiases] iter 1/50 - train_rmse=1.27827
[BayesianPMFWithBiases] iter 5/50 - train_rmse=1.27457
[BayesianPMFWithBiases] iter 10/50 - train_rmse=1.27636
[BayesianPMFWithBiases] iter 15/50 - train_rmse=1.27620
[BayesianPMFWithBiases] iter 20/50 - train_rmse=1.27600
[BayesianPMFWithBiases] iter 25/50 - train_rmse=1.27634
[BayesianPMFWithBiases] iter 30/50 - train_rmse=1.27572
[BayesianPMFWithBiases] iter 35/50 - train_rmse=1.27497
[BayesianPMFWithBiases] iter 40/50 - train_rmse=1.27447
[BayesianPMFWithBiases] iter 45/50 - train_rmse=1.27493
[BayesianPMFWithBiases] iter 50/50 - train_rmse=1.27492
MAE for BPMF with Biases Model: 1.2435700379137207


,band,n_rows,rmse,mae
0,high,11425,1.571442,1.208258
1,low,17473,1.691471,1.296288
2,mid,10138,1.658044,1.279044


In [3]:
# adaptative_prior_ensemble.save("adaptative_prior_ensemble.joblib")

adaptative_prior_ensemble = AdaptivePosteriorColdStartEnsemble.load("adaptative_prior_ensemble.joblib")

In [3]:
splitter = FrequencyBandSplitter(mode="threshold", low_threshold=1, high_threshold=10)
splitter.fit(train_data)
evaluator = FrequencyBandEvaluator(splitter)

In [23]:
knn = SurpriseKNNBaselineWrapper(
    k=30,
    min_k=6,
    user_based=False,
    sim_name="pearson_baseline",
    shrinkage=200,
    bsl_options={
        "method": "als",
        "n_epochs": 15,
        "reg_u": 3,
        "reg_i": 3,
    },
    clip_range=(1, 10),
    unknown_strategy="surprise",
    verbose=True,
    name="KNNBaselineItem"
)

# select data with higher frequency for knn
item_counts = full_data['item'].value_counts()
high_freq_items = item_counts[item_counts >= 4].index
train_data_knn = full_data[full_data['item'].isin(high_freq_items)]

knn.fit(train_data_knn)
predictions_knn = knn.predict_df(test_data, round_predictions=True)
mae_knn = np.mean(np.abs(predictions_knn['prediction'] - test_data['rating']))
print(f'MAE for KNN Baseline: {mae_knn}')

evaluator.evaluate(knn, test_data)

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
[KNNBaselineItem] entrenado | modo=item-item | k=30 | min_k=6 | sim=pearson_baseline | rating_scale=(1.0, 10.0)
MAE for KNN Baseline: 1.0605851009324725


,band,n_rows,rmse,mae
0,high,11425,1.141211,0.741927
1,low,17473,1.775530,1.365478
2,mid,10138,1.356357,0.985644


In [22]:
baseline_model = SurpriseBaselineOnlyModel()
baseline_model.fit(full_data)

baseline_predictions = baseline_model.predict_df(test_data, round_predictions=True)

mae_baseline = np.mean(np.abs(baseline_predictions['prediction'] - test_data['rating']))

print(f'MAE for Baseline Model: {mae_baseline}')

nmf = SurpriseNMFModel(n_factors=15, n_epochs=40, biased=True, verbose=False, reg_pu=0.5, reg_qi=0.5, init_low=0.001, init_high=0.001, reg_bu=0.0002, reg_bi=0.0002)
nmf.fit(full_data)

nmf_predictions = nmf.predict_df(test_data, round_predictions=True)
mae_nmf = np.mean(np.abs(nmf_predictions['prediction'] - test_data['rating']))
print(f'MAE for NMF Model: {mae_nmf}')

evaluator.evaluate(nmf, test_data)

bpmf_with_biases = BayesianPMFWithBiases(n_factors=10, n_iters=50, burn_in=20, rating_std=1, user_bias_std=0.5, item_bias_std=0.5, clip_range=(0, 10))
bpmf_with_biases.fit(full_data)

bpmf_with_biases_predictions = bpmf_with_biases.predict_df(test_data, round_predictions=True)
mae_bpmf_with_biases = np.mean(np.abs(bpmf_with_biases_predictions['prediction'] - test_data['rating']))
print(f'MAE for BPMF with Biases Model: {mae_bpmf_with_biases}')

evaluator.evaluate(bpmf_with_biases, test_data)

ensemble = RatingEnsemble(models=[baseline_model, nmf, bpmf_with_biases], weights=[0.33, 0.33, 0.34])
ensemble_predictions = ensemble.predict_df(test_data, round_predictions=True)
mae_ensemble = np.mean(np.abs(ensemble_predictions['prediction'] - test_data['rating']))
print(f'MAE for Ensemble Model: {mae_ensemble}')

evaluator.evaluate(ensemble, test_data)

MAE for Baseline Model: 1.0658366635925811
MAE for NMF Model: 0.9762014550671175
[BayesianPMFWithBiases] iter 1/50 - train_rmse=1.38624
[BayesianPMFWithBiases] iter 5/50 - train_rmse=1.38851
[BayesianPMFWithBiases] iter 10/50 - train_rmse=1.38802
[BayesianPMFWithBiases] iter 15/50 - train_rmse=1.38645
[BayesianPMFWithBiases] iter 20/50 - train_rmse=1.38503
[BayesianPMFWithBiases] iter 25/50 - train_rmse=1.38418
[BayesianPMFWithBiases] iter 30/50 - train_rmse=1.38207
[BayesianPMFWithBiases] iter 35/50 - train_rmse=1.37921
[BayesianPMFWithBiases] iter 40/50 - train_rmse=1.38038
[BayesianPMFWithBiases] iter 45/50 - train_rmse=1.37936
[BayesianPMFWithBiases] iter 50/50 - train_rmse=1.37668
MAE for BPMF with Biases Model: 0.990982682651911
MAE for Ensemble Model: 1.0047648324623424


,band,n_rows,rmse,mae
0,high,11425,1.413452,1.094420
1,low,17473,1.293050,0.995251
2,mid,10138,1.355722,1.050273


In [32]:
nmf_frequent = SurpriseNMFModel(n_factors=8, n_epochs=40, biased=True, verbose=False, reg_pu=0.5, reg_qi=0.5, init_low=0.001, init_high=0.001, reg_bu=0.0002, reg_bi=0.0002)

nmf_frequent.fit(train_data_knn)

evaluator.evaluate(nmf_frequent, test_data)

,band,n_rows,rmse,mae
0,high,11425,1.371429,1.053733
1,low,17473,1.774232,1.362287
2,mid,10138,1.421215,1.083764


In [33]:
ensemble_frequent = RatingEnsemble(models=[nmf_frequent, knn], weights=[0.5, 0.5])

In [34]:
threshold_ensemble = ThresholdItemPredictor(test_path='../data/test.csv', save_path="../data/threshold_ensemble_predictions_3.csv", 
                                            rare_model=ensemble, frequent_model=ensemble_frequent, threshold=4, train_df=full_data, round_predictions=True)

threshold_ensemble.predict()

Predictions saved to ../data/threshold_ensemble_predictions_3.csv
Errors: 0
Predictions with rare_model: 27085
Predictions with frequent_model: 16235
Unknown items in test: 16157


,ID,rating
0,0,8.0
1,1,8.0
2,2,7.0
3,3,8.0
4,4,7.0
...,...,...
43315,43315,7.0
43316,43316,8.0
43317,43317,8.0
43318,43318,7.0


In [27]:
predictor = Predictor(test_path='../data/test.csv', save_path='../data/solution_ensemble_2.csv', round_predictions=True)
predictor.predict(ensemble)

Predictions saved to ../data/solution_ensemble_2.csv. Number of errors: 0


,ID,rating
0,0,8.0
1,1,8.0
2,2,7.0
3,3,8.0
4,4,7.0
...,...,...
43315,43315,7.0
43316,43316,9.0
43317,43317,8.0
43318,43318,7.0


In [29]:
adaptative_prior_ensemble_2 = AdaptivePosteriorColdStartEnsemble(main_model=ensemble, fit_main_model=False)

adaptative_prior_ensemble_2.fit(full_data)
predictions_adaptative_prior_ensemble_2 = adaptative_prior_ensemble_2.predict_df(test_data, round_predictions=True)
mae_adaptative_prior_ensemble_2 = np.mean(np.abs(predictions_adaptative_prior_ensemble_2['prediction'] - test_data['rating']))
print(f'MAE for Adaptive Posterior Cold Start Ensemble with Ensemble as main model: {mae_adaptative_prior_ensemble_2}')

evaluator.evaluate(adaptative_prior_ensemble_2, test_data)

MAE for Adaptive Posterior Cold Start Ensemble with Ensemble as main model: 1.1728148375858183


,band,n_rows,rmse,mae
0,high,11425,1.502041,1.183423
1,low,17473,1.573797,1.247196
2,mid,10138,1.525193,1.204163


In [30]:
predictor = Predictor(test_path='../data/test.csv', save_path='../data/first_adaptative_ensemble.csv', round_predictions=True)
predictor.predict(adaptative_prior_ensemble_2)

Predictions saved to ../data/first_adaptative_ensemble.csv. Number of errors: 0


,ID,rating
0,0,8.0
1,1,8.0
2,2,7.0
3,3,8.0
4,4,7.0
...,...,...
43315,43315,7.0
43316,43316,8.0
43317,43317,8.0
43318,43318,7.0


In [ ]:
from model.baseline import SurpriseBaselineOnlyModel

baseline_model = SurpriseBaselineOnlyModel()
baseline_model.fit(train_data)

baseline_predictions = baseline_model.predict_df(test_data, round_predictions=True)

mae_baseline = np.mean(np.abs(baseline_predictions['prediction'] - test_data['rating']))

print(f'MAE for Baseline Model: {mae_baseline}')

from model.PMF import SurpriseNMFModel

nmf = SurpriseNMFModel(n_factors=15, n_epochs=40, biased=True, verbose=False, reg_pu=0.5, reg_qi=0.5, init_low=0.001, init_high=0.001, reg_bu=0.0002, reg_bi=0.0002)
nmf.fit(train_data)

nmf_predictions = nmf.predict_df(test_data, round_predictions=True)
mae_nmf = np.mean(np.abs(nmf_predictions['prediction'] - test_data['rating']))
print(f'MAE for NMF Model: {mae_nmf}')

evaluator.evaluate(nmf, test_data)

from ensemble.ensemble import RatingEnsemble

bpmf_with_biases = adaptative_prior_ensemble.main_model

ensemble = RatingEnsemble(models=[baseline_model, nmf, bpmf_with_biases], weights=[0.33, 0.33, 0.34])
ensemble_predictions = ensemble.predict_df(test_data, round_predictions=True)
mae_ensemble = np.mean(np.abs(ensemble_predictions['prediction'] - test_data['rating']))
print(f'MAE for Ensemble Model: {mae_ensemble}')

evaluator.evaluate(ensemble, test_data)

In [ ]:
predictor = Predictor(test_path='../data/test.csv', save_path='../data/solution_ensemble.csv', round_predictions=True)
predictor.predict(ensemble)